In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix,classification_report
from imblearn.over_sampling import SMOTE


## Load Dataset

In [3]:
df=pd.read_csv("Risk_Alert_Classifier_Dataset_4600 - Risk_Alert_Classifier_Dataset_4600.csv.csv")
df.head()

,customer_id,age,gender,region,employment_type,annual_income_inr,credit_score,credit_utilization_ratio,missed_payments_12m,avg_late_payment_days,monthly_transaction_count,monthly_spend_inr,cash_advance_count_6m,complaints_last_6m,failed_login_attempts_3m,account_tenure_months,last_transaction_date,debt_balance_inr,risk_status
0,500001,43.0,Female,NaN,Salaried,82242.0,NaN,0.120,1,2.2,39,33889.0,0,2,4,70,2025-09-26,87273,0
1,500002,29.0,Female,Central,Salaried,32769.0,647.0,0.337,1,1.5,11,10853.0,1,1,1,34,2025-11-24,20600,0
2,500003,36.0,Male,East,Salaried,39731.0,727.0,0.175,0,3.9,45,25519.0,2,1,1,74,2025-09-26,47565,0
3,500004,28.0,Male,North,Unemployed,38990.0,553.0,0.472,7,23.3,103,17806.0,1,2,6,72,2025-10-03,43803,1
4,500005,36.0,Female,East,Self-Employed,41043.0,732.0,0.418,1,9.8,95,27114.0,0,1,1,11,2025-10-26,12008,0


## EDA

In [4]:
df.info()
df.describe(include='all')
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer_id                4600 non-null   int64  
 1   age                        4460 non-null   float64
 2   gender                     4600 non-null   object 
 3   region                     4498 non-null   object 
 4   employment_type            4456 non-null   object 
 5   annual_income_inr          4434 non-null   float64
 6   credit_score               4384 non-null   float64
 7   credit_utilization_ratio   4453 non-null   float64
 8   missed_payments_12m        4600 non-null   int64  
 9   avg_late_payment_days      4600 non-null   float64
 10  monthly_transaction_count  4600 non-null   int64  
 11  monthly_spend_inr          4471 non-null   float64
 12  cash_advance_count_6m      4600 non-null   int64  
 13  complaints_last_6m         4600 non-null   int64

customer_id                    0
age                          140
gender                         0
region                       102
employment_type              144
annual_income_inr            166
credit_score                 216
credit_utilization_ratio     147
missed_payments_12m            0
avg_late_payment_days          0
monthly_transaction_count      0
monthly_spend_inr            129
cash_advance_count_6m          0
complaints_last_6m             0
failed_login_attempts_3m       0
account_tenure_months          0
last_transaction_date          0
debt_balance_inr               0
risk_status                    0
dtype: int64

## Identify Target & Features

In [5]:
target='risk_status'
drop_cols=[target]
for c in ['customer_id','last_transaction_date']:
    if c in df.columns:
        drop_cols.append(c)
X=df.drop(columns=drop_cols)
y=df[target]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

cat=X.select_dtypes(include='object').columns
num=X.select_dtypes(exclude='object').columns

pre=ColumnTransformer([
('num',Pipeline([('imp',KNNImputer()),('sc',StandardScaler())]),num),
('cat',Pipeline([('enc',OneHotEncoder(handle_unknown='ignore'))]),cat)
])

## Logistic Regression

In [6]:
log_pipe=Pipeline([
('pre',pre),
('model',LogisticRegression(max_iter=1000))
])
log_pipe.fit(X_train,y_train)
pred=log_pipe.predict(X_test)
print(classification_report(y_test,pred))
cm=confusion_matrix(y_test,pred)
print(cm)
tn,fp,fn,tp=cm.ravel()
print("Type I Error (FP):",fp)
print("Type II Error (FN):",fn)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       809
           1       1.00      1.00      1.00       111

    accuracy                           1.00       920
   macro avg       1.00      1.00      1.00       920
weighted avg       1.00      1.00      1.00       920

[[809   0]
 [  0 111]]
Type I Error (FP): 0
Type II Error (FN): 0


## Handle Imbalance with SMOTE

In [7]:
Xtr=pre.fit_transform(X_train)
Xte=pre.transform(X_test)
sm=SMOTE(random_state=42)
X_res,y_res=sm.fit_resample(Xtr,y_train)

## Decision Tree + Random Search + Grid Search

In [8]:
tree=DecisionTreeClassifier(random_state=42)
rand=RandomizedSearchCV(tree,{
'max_depth':[3,5,10,None],
'min_samples_split':[2,5,10],
'min_samples_leaf':[1,2,4]
},n_iter=5,cv=5,scoring='f1')
rand.fit(X_res,y_res)
grid=GridSearchCV(DecisionTreeClassifier(random_state=42),{
'max_depth':[rand.best_params_['max_depth']],
'min_samples_split':[rand.best_params_['min_samples_split']-1 if rand.best_params_['min_samples_split']>2 else 2,rand.best_params_['min_samples_split'],rand.best_params_['min_samples_split']+1],
'min_samples_leaf':[1,2,4]
},cv=5,scoring='f1')
grid.fit(X_res,y_res)
best=grid.best_estimator_
pred=best.predict(Xte)
print(classification_report(y_test,pred))
print(confusion_matrix(y_test,pred))
print(grid.best_params_)

              precision    recall  f1-score   support

           0       0.99      0.98      0.98       809
           1       0.84      0.94      0.89       111

    accuracy                           0.97       920
   macro avg       0.91      0.96      0.93       920
weighted avg       0.97      0.97      0.97       920

[[789  20]
 [  7 104]]
{'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2}


## Final Report

- Compare Logistic Regression vs Decision Tree.
- Compare before/after SMOTE.
- Discuss FP (Type I) and FN (Type II).
- Recommend the model with best Recall/F1/ROC-AUC based on business objective.
